In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = int(dbutils.widgets.get("initLoadFlag"))

### **Data Reading**

In [0]:
 df = spark.sql("SELECT * FROM databrickscatalogetep1.silver.buyerSilver")

In [0]:
df.display()

_Removing Duplicates_

In [0]:
df = df.dropDuplicates(subset=["customer_id"])

**Old seperation**

In [0]:
if init_load_flag==0:
    df_old = spark.sql('''select DimBuyerKey, customer_id , create_date , update_date
                        from databrickscatalogetep1.gold.DimBuyers ''')
    
else:
    df_old = spark.sql('''select 0 DimBuyerKey, 0 customer_id ,0 create_date , 0 update_date
                        from databrickscatalogetep1.silver.buyersilver where 1=0''')


In [0]:
df_old.display()

In [0]:
df_old = df_old.withColumnRenamed("DimBuyerKey","OldDimBuyerKey")\
                .withColumnRenamed("customer_id","OldCustomerid")\
                .withColumnRenamed("create_date","OldCreateDate")\
                .withColumnRenamed("update_date","OldUpdateDate")

In [0]:
df_old.display()

_Join with the old Records_

In [0]:
df_join =df.join(df_old, df.customer_id == df_old.OldCustomerid, "left")

**seperating new and old records**

In [0]:
df_join.display()

In [0]:
df_new =df_join.filter(df_join['OldDimBuyerKey'].isNull())

In [0]:
df_old=df_join.filter(df_join['OldDimBuyerKey'].isNotNull())

_preparing df_old_

In [0]:
df_old =df_old.drop("OldCustomerid","OldUpdateDate")

df_old =df_old.withColumnRenamed("OldDimBuyerKey","DimBuyerKey")

In [0]:
#Renaming create Date column OldCreateDate to create_date

df_old = df_old.withColumnRenamed("OldCreateDate","create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

#Recreating "update_date" column with timestamp
df_old = df_old.withColumn("update_date",current_timestamp()) 

_preparing df_new _

In [0]:
#Renaming create Date column OldCreateDate to create_date

df_new = df_new.drop("OldDimBuyerKey","OldCustomerid","OldCreateDate","OldUpdateDate")

#Recreating "update_date" column with timestamp
df_new = df_new.withColumn("update_date",current_timestamp()) 
df_new = df_new.withColumn("create_date", current_timestamp())

_Surrogate Key_

In [0]:
df_new = df_new.withColumn("DimBuyerKey",monotonically_increasing_id()+lit(1))

In [0]:
df_new.display()

_Adding max sugg key_

In [0]:
if init_load_flag==1:
    max_sugg_key=0

else:
    df_max= spark.sql('''select max(DimBuyerKey) as max_sugg_key from databrickscatalogetep1.gold.DimBuyers''')
    #Converting df_max to max_sugg_key variable
    max_sugg_key=df_max.collect()[0]['max_sugg_key']

In [0]:
df_new = df_new.withColumn("DimBuyerKey",lit(max_sugg_key)+col("DimBuyerKey"))

**Union New and old**

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

### **SCD Type 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if (spark.catalog.tableExists("databrickscatalogetep1.gold.DimBuyers")):
   dlt_obj = DeltaTable.forPath(spark,"abfss://goldstg@storageloweretep1.dfs.core.windows.net/DimBuyers")
   
   dlt_obj.alias("tgt").merge(df_final.alias("src"),"tgt.DimBuyerKey=src.DimBuyerKey")\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()
else:

    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path","abfss://goldstg@storageloweretep1.dfs.core.windows.net/DimBuyers")\
        .saveAsTable("databrickscatalogetep1.gold.DimBuyers")